In [17]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Uitzetten warnings voor opmaak
# warnings.filterwarnings('ignore')

# Opdracht 1: Voorspel Herseninfarct (Stroke Prediction)
**Competitie:** Predicting stroke 2026-27 | Kaggle  
**Teamnaam:**   Groep 4 <br>
**Teamleden & Kaggle Usernames:**  
- Kaman - [Kaggle Username 1]  
- Alex - [Kaggle Username 2]  
- Pablo - [Kaggle Username 3]  

## 1. Inleiding
In dit project werken we als junior data scientists bij *MedPredict*. Het doel is om een machine learning model te ontwikkelen dat op basis van demografische, leefstijl- en medische gegevens kan voorspellen of een patiënt een verhoogd risico loopt op een herseninfarct (stroke). Dit model zal zorgprofessionals ondersteunen bij preventie en gerichte opvolging.

In [18]:
train_df = pd.read_csv("train.csv")
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33550 entries, 0 to 33549
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              33550 non-null  int64  
 1   age                             33550 non-null  float64
 2   hypertension                    33550 non-null  int64  
 3   heart_disease                   33550 non-null  int64  
 4   avg_glucose_level               33550 non-null  float64
 5   bmi                             33550 non-null  float64
 6   gender_Female                   33550 non-null  bool   
 7   gender_Male                     33550 non-null  bool   
 8   gender_Other                    33550 non-null  bool   
 9   ever_married_No                 33550 non-null  bool   
 10  ever_married_Yes                33550 non-null  bool   
 11  work_type_Govt_job              33550 non-null  bool   
 12  work_type_Never_worked          33550 non-n

## 2. Data & Exploratieve Data Analyse (EDA)

### Uitleg van de kolommen
De dataset is al deels voorbewerkt. Categorische variabelen zijn omgezet naar dummy-kolommen (True/False).
- **id**: Uniek identificatienummer van de patiënt.
- **age**: Leeftijd van de patiënt in jaren.
- **hypertension**: 0 of 1 (of de patiënt hoge bloeddruk heeft).
- **heart_disease**: 0 of 1 (of de patiënt een hartziekte heeft).
- **avg_glucose_level**: Gemiddeld glucosegehalte in het bloed.
- **bmi**: Body Mass Index.
- **True/False kolommen (Dummies**: Gecodeerde categorische variabelen zoals `gender`, `ever_married`, `work_type`, `Residence_type`, en `smoking_status`.
- **stroke** (Target): 1 als de patiënt een herseninfarct heeft gehad, anders 0. (Ontbreekt in `test.csv`).

In [ ]:
class DataLoader:
    """Klasse voor het inladen en inspecteren van de data."""
    
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path
        
    def load_data(self):
        """Laadt de train en test datasets in."""
        # De CSV-bestanden bevatten een header-rij met kolomnamen,
        # dus we lezen ze in met de standaard header=0.
        self.train_df = pd.read_csv(self.train_path)
        self.test_df = pd.read_csv(self.test_path)
        
        # Converteer booleans (True/False) naar integers (1/0) voor Scikit-Learn
        bool_cols = self.train_df.select_dtypes(include=['bool']).columns
        for col in bool_cols:
            self.train_df[col] = self.train_df[col].astype(int)
            if col in self.test_df.columns:
                self.test_df[col] = self.test_df[col].astype(int)
                
        return self.train_df, self.test_df

# Inladen van de data
loader = DataLoader('train.csv', 'test.csv')
train_df, test_df = loader.load_data()

print("Train Data Shape:", train_df.shape)
print("Test Data Shape:", test_df.shape)
train_df.head()

Train Data Shape: (33550, 22)
Test Data Shape: (8388, 21)


,id,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,52709,30.0,0,0,63.60,33.3,0,1,0,1,...,0,1,0,0,1,0,0,0,1,0
1,72295,75.0,1,0,215.17,48.0,1,0,0,0,...,0,0,1,0,1,0,0,1,0,0
2,26451,15.0,0,0,135.22,19.0,0,1,0,1,...,0,1,0,0,1,0,0,1,0,0
3,65210,47.0,0,0,64.89,28.2,0,1,0,0,...,0,0,1,0,0,1,0,1,0,0
4,69299,49.0,0,0,222.34,28.8,0,1,0,0,...,0,0,1,0,1,0,1,0,0,0
